In [ ]:
import pandas as pd
import numpy as np

# =========================================================
# 1. LOAD DATA
# =========================================================
# Replace with your actual file path
CSV_PATH = "chemtastes_smiles.csv"

df = pd.read_csv(CSV_PATH)

# Ensure expected columns exist
required_cols = ["canonical SMILES", "Taste", "Name"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Dataset loaded successfully")
print("Total rows:", df.shape[0])
print("Unique SMILES:", df["canonical SMILES"].nunique())

# =========================================================
# 2. DEFINE TASTE KEYWORDS (ROBUST + BROAD)
# =========================================================
# Rule: if it appears EVEN SLIGHTLY → count it
TASTE_KEYWORDS = {
    "Sweet":  ["sweet"],
    "Bitter": ["bitter"],
    "Umami":  ["umami", "msg", "brothy", "kokumi"],
    "Sour":   ["sour"],
    "Salty":  ["salty"]
}

# =========================================================
# 3. MAP A SINGLE TASTE STRING → MULTI-LABEL VECTOR
# =========================================================
def map_tastes(taste_text):
    """
    Convert raw Taste description into binary labels:
    Sweet, Bitter, Umami, Sour, Salty
    """
    # Handle missing safely
    if pd.isna(taste_text):
        return pd.Series({
            "Sweet": 0,
            "Bitter": 0,
            "Umami": 0,
            "Sour": 0,
            "Salty": 0
        })

    text = taste_text.lower()

    labels = {}
    for taste, keywords in TASTE_KEYWORDS.items():
        labels[taste] = int(
            any(keyword in text for keyword in keywords)
        )

    return pd.Series(labels)

# =========================================================
# 4. APPLY MAPPING TO EACH ROW (ROW-LEVEL LABELS)
# =========================================================
taste_labels = df["Taste"].apply(map_tastes)

df_labeled = pd.concat([df, taste_labels], axis=1)

print("\nTaste mapping completed.")
print(df_labeled[["Taste", "Sweet", "Bitter", "Umami", "Sour", "Salty"]].head())

# =========================================================
# 5. AGGREGATE DUPLICATES BY CANONICAL SMILES (LOGICAL OR)
# =========================================================
# If ANY duplicate row has a taste → molecule has that taste
df_smiles_level = (
    df_labeled
    .groupby("canonical SMILES")[["Sweet", "Bitter", "Umami", "Sour", "Salty"]]
    .max()
    .reset_index()
)

# Attach a representative name (first occurrence)
name_map = (
    df.groupby("canonical SMILES")["Name"]
    .first()
    .reset_index()
)

df_final = df_smiles_level.merge(
    name_map, on="canonical SMILES", how="left"
)

print("\nDuplicate aggregation completed.")
print("Final molecule-level rows:", df_final.shape[0])

# =========================================================
# 6. SANITY CHECKS (IMPORTANT)
# =========================================================
# Number of tastes per molecule
df_final["num_tastes"] = df_final[
    ["Sweet", "Bitter", "Umami", "Sour", "Salty"]
].sum(axis=1)

print("\nNumber of tastes per molecule:")
print(df_final["num_tastes"].value_counts().sort_index())

# Check molecules with multiple tastes
print("\nExamples of multi-taste molecules:")
print(df_final[df_final["num_tastes"] >= 2].head(10))

# Verify umami detection
print("\nSample Umami molecules:")
print(df_final[df_final["Umami"] == 1].head(10))

# =========================================================
# 7. SAVE FINAL CLEAN DATASET
# =========================================================
OUTPUT_PATH = "chemtastesdb_multilabel_clean.csv"
df_final.to_csv(OUTPUT_PATH, index=False)

print(f"\nFinal multi-label dataset saved to: {OUTPUT_PATH}")


ValueError: Missing required columns: ['canonical SMILES', 'Name']

In [ ]:
import re
import pandas as pd

PRIMARY_TASTES = ["sweet", "bitter", "umami", "sour", "salty"]

def parse_taste_string(taste_text):
    """
    Parse a single ChemTastesDB taste string into
    presence/absence of basic tastes, handling negations correctly.
    
    Returns a dict for easy inspection.
    """
    result = {t.capitalize(): 0 for t in PRIMARY_TASTES}

    if pd.isna(taste_text):
        return result

    text = taste_text.lower()

    # Split on common separators
    tokens = re.split(r"[;,]", text)

    for token in tokens:
        token = token.strip()

        for taste in PRIMARY_TASTES:
            neg_pattern = rf"\bnon[- ]?{taste}\b"
            pos_pattern = rf"\b{taste}\b"

            # Explicit negation → force 0
            if re.search(neg_pattern, token):
                continue

            # Positive mention
            if re.search(pos_pattern, token):
                result[taste.capitalize()] = 1

    return result

for desc in df["Taste"].unique():
    print(desc+ " --> ", end="")
    for (taste, presence) in parse_taste_string(desc).items():
        if presence:
            print(f"{taste}, ", end="")
    print()

# Sweetish -->
# Bitter/Non-bitter; Sweet(5) --> Sweet, 
# Sourness --> 
# Non-sweet/Sweet; Tasteless --> 
# Not-sweet --> Sweet, 
# Sweet/Non-sweet --> 
# Non-sweet/Sweet --> 
# Bitter/Non-bitter --> 



Sweet --> Sweet, 
Non-bitter; Sweet(6) --> Sweet, 
Non-bitter; Sweet --> Sweet, 
Sweetish --> 
Low sweet; Sweet --> Sweet, 
Sweet(8); Intensely sweet --> Sweet, 
Sweet; Less sweet --> Sweet, 
Slightly sweet --> Sweet, 
Sweet(6); Sweet/Bitter --> Sweet, Bitter, 
Sweet; Very sweet --> Sweet, 
Tasteless; Sweet(2) --> Sweet, 
Sweet  --> Sweet, 
Sweet(2); Non-bitter --> Sweet, 
Sweet(3); Non-bitter --> Sweet, 
Sweet(5); Non-bitter --> Sweet, 
Tasteless; Sweet(3) --> Sweet, 
Sweet(4); Intensely sweet(2) --> Sweet, 
Sweet(2); Less sweet --> Sweet, 
Slightly sweet; Sweet(2) --> Sweet, 
Very sweet; Sweet --> Sweet, 
Non-bitter; Sweet(3) --> Sweet, 
Intensely sweet --> Sweet, 
Very sweet; Sweet(3) --> Sweet, 
Non-bitter; Sweet(5) --> Sweet, 
Sweet(3); Sweetish --> Sweet, 
Sweet(5); Bitter --> Sweet, Bitter, 
Sweet(17); Bitter; Sweet/Bitter; Sweet (bitter aftertaste) --> Sweet, Bitter, 
Sweet(11); Bitter --> Sweet, Bitter, 
Intensely sweet; Sweet(4) --> Sweet, 
Sweet(35); Sweet/Bitter --> Sweet, 

# Sweetish -->
# Bitter/Non-bitter; Sweet(5) --> Sweet, 
# Sourness --> 
# Non-sweet/Sweet; Tasteless --> 
# Not-sweet --> Sweet, 
# Sweet/Non-sweet --> 
# Non-sweet/Sweet --> 
# Bitter/Non-bitter --> 


In [ ]:
target = {}
for desc in df["Taste"].unique():
    target[desc] = parse_taste_string(desc)

# -------------------------------
# MANUAL OVERRIDES (FINAL)
# -------------------------------

target["Sweetish"] = {
    'Sweet': 1, 'Bitter': 0, 'Umami': 0, 'Sour': 0, 'Salty': 0
}

target["Bitter/Non-bitter; Sweet(5)"] = {
    'Sweet': 1, 'Bitter': 1, 'Umami': 0, 'Sour': 0, 'Salty': 0
}

target["Sourness"] = {
    'Sweet': 0, 'Bitter': 0, 'Umami': 0, 'Sour': 1, 'Salty': 0
}

target["Non-sweet/Sweet; Tasteless"] = {
    'Sweet': 1, 'Bitter': 0, 'Umami': 0, 'Sour': 0, 'Salty': 0
}

target["Not-sweet"] = {
    'Sweet': 0, 'Bitter': 0, 'Umami': 0, 'Sour': 0, 'Salty': 0
}

target["Sweet/Non-sweet"] = {
    'Sweet': 1, 'Bitter': 0, 'Umami': 0, 'Sour': 0, 'Salty': 0
}

target["Non-sweet/Sweet"] = {
    'Sweet': 1, 'Bitter': 0, 'Umami': 0, 'Sour': 0, 'Salty': 0
}

target["Bitter/Non-bitter"] = {
    'Sweet': 0, 'Bitter': 1, 'Umami': 0, 'Sour': 0, 'Salty': 0
}


In [ ]:
import json

with open("taste_mapping.json", "w", encoding="utf-8") as f:
    json.dump(target, f, indent=4)

In [ ]:
df["Class taste"].unique()

array(['Sweetness', 'Bitterness', 'Umaminess', 'Sourness', 'Saltiness',
       'Multitaste', 'Tastelessness', 'Non-sweetness', 'Non-bitterness',
       'Miscellaneous'], dtype=object)

In [ ]:
CLASS_TASTE_MAP = {
    "Sweetness":     {"Sweet": 1, "Bitter": 0, "Umami": 0, "Sour": 0, "Salty": 0},
    "Bitterness":    {"Sweet": 0, "Bitter": 1, "Umami": 0, "Sour": 0, "Salty": 0},
    "Umaminess":     {"Sweet": 0, "Bitter": 0, "Umami": 1, "Sour": 0, "Salty": 0},
    "Sourness":      {"Sweet": 0, "Bitter": 0, "Umami": 0, "Sour": 1, "Salty": 0},
    "Saltiness":     {"Sweet": 0, "Bitter": 0, "Umami": 0, "Sour": 0, "Salty": 1},
    "Tastelessness": {"Sweet": 0, "Bitter": 0, "Umami": 0, "Sour": 0, "Salty": 0},
    "Non-sweetness": {"Sweet": 0, "Bitter": 0, "Umami": 0, "Sour": 0, "Salty": 0},
    "Non-bitterness":{"Sweet": 0, "Bitter": 0, "Umami": 0, "Sour": 0, "Salty": 0},
    "Multitaste":    {"Sweet": 1, "Bitter": 1, "Umami": 1, "Sour": 1, "Salty": 1},
    "Miscellaneous": {"Sweet": 0, "Bitter": 0, "Umami": 0, "Sour": 0, "Salty": 0},
}
def resolve_row_taste(row, target_dict):
    """
    Priority:
    1. Exact Taste string lookup in target dict
    2. Fallback to Class taste
    """
    taste_str = row["Taste"]
    class_taste = row["Class taste"]

    if taste_str in target_dict:
        return target_dict[taste_str]

    if class_taste in CLASS_TASTE_MAP:
        return CLASS_TASTE_MAP[class_taste]

    # Absolute fallback (should be rare)
    return {"Sweet": 0, "Bitter": 0, "Umami": 0, "Sour": 0, "Salty": 0}



row_targets = df.apply(
    lambda row: resolve_row_taste(row, target),
    axis=1
)

row_targets_df = pd.DataFrame(list(row_targets))
df_with_targets = pd.concat([df, row_targets_df], axis=1)
FINAL_TASTES = ["Sweet", "Bitter", "Umami", "Sour", "Salty"]

df_final = (
    df_with_targets
    .groupby("canonical SMILES")[FINAL_TASTES]
    .max()          # logical OR for binary columns
    .reset_index()
)


In [ ]:
df.describe()

,taste_word_count,smiles_length,num_carbons,num_oxygens,num_nitrogens,num_sulfurs,num_halogens,num_rings,num_aromatic,num_double_bonds,num_triple_bonds,num_brackets,num_branches,C_O_ratio,N_C_ratio,ring_aromatic_ratio,heteroatom_ratio
count,4075.000000,4075.000000,4075.000000,4075.00000,4075.000000,4075.000000,4075.000000,4075.000000,4075.000000,4075.000000,4075.000000,4075.000000,4075.000000,4075.000000,4075.000000,4075.000000,4075.000000
mean,1.223067,45.932761,12.482945,5.84319,1.845890,0.266503,0.209080,4.065521,4.345031,2.866748,0.039755,0.826994,11.941104,1.919406,0.204609,0.861872,0.168038
std,0.735560,33.799837,12.939984,5.39884,2.902998,0.545562,0.676389,4.012178,5.392655,2.718987,0.202803,1.855946,11.201220,1.594294,0.286916,0.948810,0.056317
min,1.000000,2.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,29.000000,5.000000,3.00000,0.000000,0.000000,0.000000,2.000000,0.000000,2.000000,0.000000,0.000000,6.000000,1.000000,0.000000,0.000000,0.133333
50%,1.000000,37.000000,9.000000,4.00000,1.000000,0.000000,0.000000,2.000000,0.000000,2.000000,0.000000,0.000000,8.000000,1.666667,0.153846,0.000000,0.166667
75%,1.000000,49.000000,15.000000,7.00000,2.000000,0.000000,0.000000,6.000000,6.000000,4.000000,0.000000,0.000000,14.000000,2.543561,0.277778,2.000000,0.203125
max,14.000000,563.000000,181.000000,56.00000,50.000000,4.000000,6.000000,24.000000,60.000000,50.000000,2.000000,16.000000,184.000000,40.000000,4.000000,2.608696,0.666667


In [34]:
df=pd.read_csv("chemtastes_smiles.csv")
df.describe()

,SMILES,Class taste,Taste
count,4075,4075,4075
unique,3849,10,341
top,OCC1OC(O)C(O)C(O)C1O,Bitterness,Bitter
freq,7,1615,1396


In [35]:
CLASS_TASTE_MAP = {
    "Sweetness":     {"Sweet": 1, "Bitter": 0, "Umami": 0, "Sour": 0, "Salty": 0},
    "Bitterness":    {"Sweet": 0, "Bitter": 1, "Umami": 0, "Sour": 0, "Salty": 0},
    "Umaminess":     {"Sweet": 0, "Bitter": 0, "Umami": 1, "Sour": 0, "Salty": 0},
    "Sourness":      {"Sweet": 0, "Bitter": 0, "Umami": 0, "Sour": 1, "Salty": 0},
    "Saltiness":     {"Sweet": 0, "Bitter": 0, "Umami": 0, "Sour": 0, "Salty": 1},
    "Tastelessness": {"Sweet": 0, "Bitter": 0, "Umami": 0, "Sour": 0, "Salty": 0},
    "Non-sweetness": {"Sweet": 0, "Bitter": 0, "Umami": 0, "Sour": 0, "Salty": 0},
    "Non-bitterness":{"Sweet": 0, "Bitter": 0, "Umami": 0, "Sour": 0, "Salty": 0},
    "Multitaste":    {"Sweet": 1, "Bitter": 1, "Umami": 1, "Sour": 1, "Salty": 1},
    "Miscellaneous": {"Sweet": 0, "Bitter": 0, "Umami": 0, "Sour": 0, "Salty": 0},
}
def resolve_row_taste(row, target_dict):
    """
    Priority:
    1. Exact Taste string lookup in target dict
    2. Fallback to Class taste
    """
    taste_str = row["Taste"]
    class_taste = row["Class taste"]

    if taste_str in target_dict:
        return target_dict[taste_str]

    if class_taste in CLASS_TASTE_MAP:
        return CLASS_TASTE_MAP[class_taste]

    # Absolute fallback (should be rare)
    return {"Sweet": 0, "Bitter": 0, "Umami": 0, "Sour": 0, "Salty": 0}



row_targets = df.apply(
    lambda row: resolve_row_taste(row, target),
    axis=1
)

row_targets_df = pd.DataFrame(list(row_targets))
df_with_targets = pd.concat([df, row_targets_df], axis=1)
FINAL_TASTES = ["Sweet", "Bitter", "Umami", "Sour", "Salty"]

df_final = (
    df_with_targets
    .groupby("SMILES")[FINAL_TASTES]
    .max()          # logical OR for binary columns
    .reset_index()
)


In [36]:
df_final["num_tastes"] = df_final[
    ["Sweet", "Bitter", "Umami", "Sour", "Salty"]
].sum(axis=1)

df_final = df_final.rename(columns={"SMILES": "canonical SMILES"})

df_final.head(5)


,canonical SMILES,Sweet,Bitter,Umami,Sour,Salty,num_tastes
0,Br.Br.CC1C2CCC3C4CC=C5CC(CCC5(C)C4CCC23CN1C)N(C)C,0,1,0,0,0,1
1,Br.COC(=O)C1=CCCN(C)C1,0,1,0,0,0,1
2,BrCCN1C(=O)c2ccccc2S1(=O)=O,0,0,0,0,0,0
3,Brc1ccc2c(c1)S(=O)(=O)NC2=O,1,1,0,0,0,2
4,Brc1cnc(nc1)C#N,1,0,0,0,0,1


In [38]:
OUTPUT_PATH = "chemtastesdb_multilabel_clean.csv"
df_final.to_csv(OUTPUT_PATH, index=False)

In [44]:
df_final["num_tastes"].value_counts().sort_index()



num_tastes
0     562
1    3001
2     266
3      17
4       2
5       1
Name: count, dtype: int64

# MULTITASTED   